# Instalación de librerías y comprobación de entorno


In [ ]:
%%capture
# Desinstalar versiones conflictivas si las hay
!pip uninstall -y torch torchvision torchaudio bitsandbytes transformers peft trl

# PyTorch con soporte CUDA 12.1 (compatible con Colab T4 y A10G)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Stack de fine-tuning
!pip install \
    transformers==4.46.3 \
    tokenizers \
    accelerate \
    peft==0.13.2 \
    trl==0.12.1 \
    bitsandbytes==0.44.1 \
    datasets

# Métricas y tracking
!pip install scikit-learn mlflow python-dotenv pandas

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# En Colab: subir functions.py a /content/ junto con grading_dataset.jsonl
sys.path.insert(0, "/content")
from functions import (
    LABEL_RELEVANTE, LABEL_NO_RELEVANTE, LABELS, GRADING_SYSTEM_PROMPT,
    check_gpu, load_grading_dataset, split_dataset, show_split_stats,
    build_grading_messages, format_training_prompt, examples_to_hf_dataset,
    load_tokenizer, load_model_4bit, build_peft_model, get_training_args,
    run_training, save_adapter, predict_relevance, evaluate_model,
    print_comparison, log_experiment,
)

check_gpu()

In [ ]:
# ============================================================
# MONTAR GOOGLE DRIVE (para guardar el modelo de forma persistente)
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# CONFIGURACIÓN DE RUTAS Y CONSTANTES
# ============================================================

# Dataset — copiar grading_dataset.jsonl al root de Colab antes de ejecutar
DATASET_PATH = Path("/content/grading_dataset.jsonl")

LABEL_RELEVANTE    = "relevante"
LABEL_NO_RELEVANTE = "no relevante"
LABELS = [LABEL_RELEVANTE, LABEL_NO_RELEVANTE]

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Salida en Google Drive — persiste entre sesiones de Colab
DRIVE_BASE   = Path("/content/drive/MyDrive/normabot")
OUTPUT_DIR   = str(DRIVE_BASE / "qwen-grader-lora")
ADAPTER_PATH = str(DRIVE_BASE / "qwen-grader-lora" / "adapter_final")
MERGED_PATH  = str(DRIVE_BASE / "qwen-grader-merged")   # solo si se hace merge

MAX_SEQ_LEN = 512

print("Configuración:")
print(f"  Modelo base:    {MODEL_ID}")
print(f"  Dataset:        {DATASET_PATH}  (existe: {DATASET_PATH.exists()})")
print(f"  Output dir:     {OUTPUT_DIR}")
print(f"  Adapter path:   {ADAPTER_PATH}")
print(f"  Max seq len:    {MAX_SEQ_LEN}")

In [ ]:
all_data = load_grading_dataset(DATASET_PATH)

In [ ]:
train_data, val_data, test_data = split_dataset(all_data)

# Evaluación baseline: modelo base sin fine-tunear

Antes de entrenar, evaluamos el rendimiento del modelo base con **prompting directo**.
Esto establece el punto de comparación para medir el impacto del fine-tuning.

El prompt de grading que usamos aquí es equivalente al que ya usa `src/rag/main.py`,
pero adaptado para producir `"relevante"` / `"no relevante"` en lugar de `"si"` / `"no"`.


In [ ]:
tokenizer  = load_tokenizer(MODEL_ID)
base_model = load_model_4bit(MODEL_ID)

In [ ]:
sample = test_data[0]
pred = predict_relevance(sample["query"], sample["document"], base_model, tokenizer)
print("Prediccion baseline (muestra):")
print(f"  Query:    {sample['query'][:80]}...")
print(f"  Predicho: {pred}")
print(f"  Real:     {sample['label']}")

In [ ]:
baseline_acc, baseline_f1 = evaluate_model(
    test_data, base_model, tokenizer,
    name="BASELINE — Qwen 2.5 3B-Instruct (sin fine-tuning)"
)

# Evaluación del modelo fine-tuneado

Evaluamos el modelo con los mismos ejemplos de test que usamos para el baseline,
para poder comparar directamente ambos enfoques.


In [ ]:
model.eval()
finetuned_acc, finetuned_f1 = evaluate_model(
    test_data, model, tokenizer,
    name="FINE-TUNED — Qwen 2.5 3B + QLoRA"
)

In [ ]:
mejora_acc, mejora_f1 = print_comparison(
    baseline_acc, baseline_f1, finetuned_acc, finetuned_f1
)

# Integración con el orquestador (`src/rag/main.py`)

El modelo fine-tuneado reemplaza al modelo base en la función `grade()` de `src/rag/main.py`.
La salida cambia de `"si"`/`"no"` a `"relevante"`/`"no relevante"` para mayor claridad.

## Opciones de integración

| Opción | Cuándo usarla |
|--------|---------------|
| **A — Carga directa PEFT** | Desarrollo, testing, entornos con GPU propia |
| **B — Convertir a GGUF + Ollama** | Producción (mismo setup que el modelo actual) |

El fragmento de código de abajo muestra la **Opción A**.
Para la Opción B, hacer merge del adaptador (celda anterior), convertir con `llama.cpp`,
y actualizar el `model` en `_get_grading_llm()` a `qwen2.5-grader:3b`.


In [ ]:
# ============================================================
# Fragmento de integración para src/rag/main.py
# ============================================================
# Copiar este código en src/rag/main.py para activar el grader fine-tuneado.
# Requiere que el adaptador esté en FINETUNED_ADAPTER_PATH.
# ============================================================

INTEGRATION_SNIPPET = '''
# ---- Añadir en src/rag/main.py ----

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

FINETUNED_ADAPTER_PATH = "src/finetuning/output/qwen-grader-lora/adapter_final"

GRADING_SYSTEM_PROMPT_FT = (
    "Eres un asistente especializado en normativa de inteligencia artificial. "
    "Tu tarea es determinar si un documento contiene información útil para responder "
    "una consulta sobre regulación de IA (EU AI Act, BOE, normativa española). "
    "Responde únicamente con \'relevante\' o \'no relevante\', sin explicación adicional."
)

_grading_model_ft = None
_grading_tok_ft   = None


def _get_grading_model_finetuned():
    """Carga el grader fine-tuneado (Qwen 2.5 3B + LoRA) — singleton."""
    global _grading_model_ft, _grading_tok_ft
    if _grading_model_ft is None:
        base = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen2.5-3B-Instruct",
            device_map="auto",
            torch_dtype=torch.float16,
        )
        _grading_model_ft = PeftModel.from_pretrained(base, FINETUNED_ADAPTER_PATH)
        _grading_model_ft.eval()
        _grading_tok_ft = AutoTokenizer.from_pretrained(FINETUNED_ADAPTER_PATH)
    return _grading_model_ft, _grading_tok_ft


def grade_with_finetuned(query: str, docs: list[dict], threshold: float = 0.7) -> list[dict]:
    """
    Versión actualizada de grade() que usa el modelo fine-tuneado.
    Salida: "relevante" / "no relevante" (en lugar de "si" / "no").
    Fallback a filtro por score si el modelo no está disponible.
    """
    if not docs:
        return []

    try:
        model_ft, tok_ft = _get_grading_model_finetuned()
    except Exception:
        logger.warning("Grader fine-tuneado no disponible, usando fallback por score")
        return _grade_by_score(docs, threshold)

    relevant = []
    for doc in docs:
        messages = [
            {"role": "system", "content": GRADING_SYSTEM_PROMPT_FT},
            {"role": "user", "content": (
                f"Consulta: {query}\\n\\n"
                f"Documento: {doc[\'doc\']}\\n\\n"
                "¿Es este documento relevante para responder la consulta?"
            )},
        ]
        text   = tok_ft.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok_ft(text, return_tensors="pt").to(model_ft.device)

        with torch.no_grad():
            out = model_ft.generate(**inputs, max_new_tokens=12, do_sample=False,
                                    pad_token_id=tok_ft.eos_token_id)
        gen      = out[0][inputs["input_ids"].shape[1]:]
        response = tok_ft.decode(gen, skip_special_tokens=True).strip().lower()

        if "no relevante" not in response and "relevante" in response:
            relevant.append(doc)

    return relevant
'''

print(INTEGRATION_SNIPPET)

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

print(f"Verificando adaptador en: {ADAPTER_PATH}")
del model
torch.cuda.empty_cache()

base_for_test = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map='auto', torch_dtype=torch.float16, trust_remote_code=True
)
loaded_model = PeftModel.from_pretrained(base_for_test, ADAPTER_PATH)
loaded_model.eval()
loaded_tok = load_tokenizer(ADAPTER_PATH)

print("\nPruebas de inferencia con modelo recargado desde Drive:")
for ex in [test_data[0], test_data[-1]]:
    pred = predict_relevance(ex["query"], ex["document"], loaded_model, loaded_tok)
    ok = pred == ex["label"]
    print(f"  {'OK' if ok else 'FAIL'}  Real: {ex['label']:<15} | Predicho: {pred}")

print("\nAdaptador cargado y funcionando correctamente.")